In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 1. Load the model and tokenizer
# Use the Phi-3-mini-4k-instruct model from Microsoft.
# loading model from local
model_id = "./sml_ins"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the model with 4-bit quantization to reduce memory usage.
# This makes it possible to run on standard hardware.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.backends.mps.is_available() else torch.float32,
    device_map="auto"
)

/Users/sathishkumarchandran/IdeaProjects/llm/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


In [2]:
# 2. Define the text to be summarized
long_text = """
The Amazon rainforest is a moist broadleaf forest covering most of the Amazon basin of South America.
This basin encompasses 7,000,000 square kilometres (2,700,000 sq mi), of which 5,500,000 square kilometres (2,100,000 sq mi) are covered by the rainforest.
This region includes territory belonging to nine nations. The majority of the forest is contained within Brazil, with 60% of the rainforest.
The remaining 40% is spread across Peru, Colombia, Venezuela, Ecuador, Bolivia, Guyana, Suriname, and French Guiana.
The Amazon represents over half of the planet's remaining rainforests and comprises the largest and most biodiverse tract of tropical rainforest in the world.
It is an important global carbon sink, playing a vital role in regulating the Earth's climate. However, deforestation rates in the Amazon have accelerated in recent years due to agricultural expansion and logging, posing a significant threat to its delicate ecosystem and contributing to climate change.
"""

# 3. Create the prompt for summarization
# This is an instruction-based prompt, typical for an "instruct" model variant.
prompt = f"""
### Instruction:
Summarize the following text in one concise paragraph.

### Input:
{long_text}

### Response:
"""

# 4. Use the pipeline for text generation
# The pipeline handles the tokenization and model generation process for us.
text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

# 5. Generate and print the summary
# The `max_new_tokens` and `do_sample` parameters control the output.
outputs = text_generator(
    prompt,
    max_new_tokens=100,  # Max length of the generated summary
    do_sample=False,     # For consistent, deterministic output
)

summary = outputs[0]['generated_text'].strip()

# Print only the generated response portion
if '### Response:' in summary:
    summary = summary.split('### Response:')[1].strip()

print("Original Text:\n", long_text)
print("\nGenerated Summary:\n", summary)


Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Original Text:
 
The Amazon rainforest is a moist broadleaf forest covering most of the Amazon basin of South America.
This basin encompasses 7,000,000 square kilometres (2,700,000 sq mi), of which 5,500,000 square kilometres (2,100,000 sq mi) are covered by the rainforest.
This region includes territory belonging to nine nations. The majority of the forest is contained within Brazil, with 60% of the rainforest.
The remaining 40% is spread across Peru, Colombia, Venezuela, Ecuador, Bolivia, Guyana, Suriname, and French Guiana.
The Amazon represents over half of the planet's remaining rainforests and comprises the largest and most biodiverse tract of tropical rainforest in the world.
It is an important global carbon sink, playing a vital role in regulating the Earth's climate. However, deforestation rates in the Amazon have accelerated in recent years due to agricultural expansion and logging, posing a significant threat to its delicate ecosystem and contributing to climate change.


Ge

In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from datasets import Dataset
# --- STEP 1: Prepare the training data ---
data = {
    "text": [
        "Instruction: Segment the following feedback into a summary.\nInput: The attorney demonstrated excellent drafting skills in complex agreements but occasionally missed minor clauses related to confidentiality.\nOutput: The attorney drafted complex agreements well but sometimes overlooked confidentiality clauses.",

        "Instruction: Segment the following feedback into a summary.\nInput: The paralegal showed initiative in gathering evidence and coordinating with witnesses, ensuring all materials were ready ahead of schedule.\nOutput: The paralegal proactively managed evidence and witness coordination ahead of schedule.",

        "Instruction: Segment the following feedback into a summary.\nInput: The legal consultant provided thorough compliance reviews but failed to adapt quickly when regulations changed mid-project.\nOutput: The consultant was thorough in compliance reviews but slow to adjust to regulatory changes.",

        "Instruction: Segment the following feedback into a summary.\nInput: The associate handled client communication effectively and resolved queries quickly but struggled with courtroom confidence.\nOutput: The associate communicated well with clients but lacked confidence in court.",

        "Instruction: Segment the following feedback into a summary.\nInput: The firm’s billing process lacked transparency, though the quality of legal advice was consistently praised by clients.\nOutput: The firm’s legal advice was strong, but billing transparency needs improvement.",

        "Instruction: Segment the following feedback into a summary.\nInput: The senior counsel provided insightful strategic guidance during arbitration but relied heavily on junior staff for documentation.\nOutput: The senior counsel offered strong strategic guidance but depended on juniors for documentation.",

        "Instruction: Segment the following feedback into a summary.\nInput: The junior lawyer displayed strong research abilities yet failed to meet internal deadlines on multiple occasions.\nOutput: The junior lawyer excelled in research but struggled to meet deadlines.",

        "Instruction: Segment the following feedback into a summary.\nInput: The contract review was comprehensive, and all risk factors were identified early, demonstrating strong analytical skills.\nOutput: The contract review was thorough and demonstrated strong analytical ability.",

        "Instruction: Segment the following feedback into a summary.\nInput: The litigation team collaborated well under pressure and achieved a favorable outcome for the client despite tight timelines.\nOutput: The litigation team worked efficiently under pressure to secure a favorable outcome.",

        "Instruction: Segment the following feedback into a summary.\nInput: The legal partner managed negotiations assertively, ensuring favorable contract terms for the client while maintaining professionalism.\nOutput: The partner negotiated assertively and secured favorable contract terms professionally."
    ]
}

train_dataset = Dataset.from_dict(data)

# --- STEP 2: Load the SLM and tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the model with 4-bit quantization to reduce memory usage.
# This makes it possible to run on standard hardware.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.backends.mps.is_available() else torch.float32,
    device_map="auto"
)
tokenizer.pad_token = tokenizer.eos_token

# --- STEP 3: Configure and apply LoRA ---
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# --- STEP 4: Train the model ---
training_args = TrainingArguments(
    output_dir="./lora_finetuning_output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=50,
    fp16=True,  # Set this to True
    bf16=False
)

from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    dataset_text_field="text",
    fp16=True,
    bf16=False,
    # max_seq_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    # tokenizer=tokenizer,
    args=training_args,
    formatting_func=lambda ex: ex["text"],
)


trainer.train()

# --- STEP 5: Test the fine-tuned model ---
test_prompt = "Instruction: Segment the following feedback into a summary. \nInput: Mark consitently provided good advice both legal and business.  He is always willing to bring in further experise when necessary.\nOutput:"

inputs = tokenizer(test_prompt, return_tensors="pt").to("cpu")
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    num_return_sequences=1
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)


trainable params: 921,600 || all params: 135,436,608 || trainable%: 0.6805


Truncating train dataset: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 2043.68 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
/Users/sathishkumarchandran/IdeaProjects/llm/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


Instruction: Segment the following feedback into a summary. 
Input: Mark consitently provided good advice both legal and business.  He is always willing to bring in further experise when necessary.
Output:


In [5]:
train_dataset

Dataset({
    features: ['text'],
    num_rows: 3
})

In [16]:
!pip install -U trl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [29]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import Dataset
from peft import LoraConfig, get_peft_model

# ---------------------------------------------------------
# 1️⃣  Data
# ---------------------------------------------------------
data = data = {
    "prompt": [
        "Instruction: Segment the following feedback into a summary.\nInput: The attorney demonstrated a strong understanding of contract law but occasionally failed to communicate updates in a timely manner.\nOutput:",
        "Instruction: Segment the following feedback into a summary.\nInput: The legal team efficiently handled all documentation but needed more strategic foresight during negotiations.\nOutput:",
        "Instruction: Segment the following feedback into a summary.\nInput: Counsel provided clear guidance throughout the litigation process and ensured that all filings were accurate and on time.\nOutput:",
        "Instruction: Segment the following feedback into a summary.\nInput: While the associate’s drafting skills were excellent, they required supervision to meet court deadlines.\nOutput:",
        "Instruction: Segment the following feedback into a summary.\nInput: The client appreciated the lawyer’s professionalism but mentioned delays in response to urgent emails.\nOutput:",
        "Instruction: Segment the following feedback into a summary.\nInput: The paralegal showed great initiative in gathering evidence and coordinating witnesses for the trial.\nOutput:",
        "Instruction: Segment the following feedback into a summary.\nInput: The firm provided high-quality service and practical advice but the billing transparency could be improved.\nOutput:",
        "Instruction: Segment the following feedback into a summary.\nInput: The attorney was well-prepared for hearings, displaying confidence and mastery of case facts.\nOutput:",
        "Instruction: Segment the following feedback into a summary.\nInput: The legal consultant offered valuable insights on compliance matters but lacked attention to administrative details.\nOutput:",
        "Instruction: Segment the following feedback into a summary.\nInput: The senior partner’s strategic decisions significantly strengthened the client’s negotiation position.\nOutput:"
    ],
    "response": [
        "The attorney was knowledgeable in contract law but sometimes slow to communicate updates.",
        "The team managed documentation well but lacked negotiation strategy.",
        "Counsel provided accurate and timely litigation guidance.",
        "The associate wrote well but needed oversight to meet deadlines.",
        "The lawyer was professional but slow in replying to urgent messages.",
        "The paralegal was proactive in organizing trial evidence and witnesses.",
        "The firm delivered quality service and advice but lacked billing transparency.",
        "The attorney was confident and well-prepared for hearings.",
        "The consultant was insightful on compliance but missed minor administrative details.",
        "The senior partner’s strategy improved the client’s negotiation position."
    ]
}

dataset = Dataset.from_dict(data)

# ---------------------------------------------------------
# 2️⃣  Model and Tokenizer
# ---------------------------------------------------------
# model_id = "HuggingFaceH4/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map="auto"
)

# ---------------------------------------------------------
# 3️⃣  Apply LoRA
# ---------------------------------------------------------
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# ---------------------------------------------------------
# 4️⃣  Tokenization with label masking
# ---------------------------------------------------------
def tokenize_function(example):
    # Combine instruction and response
    full_prompt = example["prompt"] + " " + example["response"]

    tokenized = tokenizer(
        full_prompt,
        truncation=True,
        padding="max_length",
        max_length=256
    )
    # Use same tokens as labels — no masking
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized


tokenized_dataset = dataset.map(tokenize_function)

# ---------------------------------------------------------
# 5️⃣  Training
# ---------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./smollm2_lora_output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=8,     # ⬆️ train longer
    logging_steps=10,
    save_steps=50,
    fp16=False,
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)
trainer.train()

# ---------------------------------------------------------
# 6️⃣  Test generation
# ---------------------------------------------------------
prompt = (
    "Instruction: Segment the following feedback into a summary.\n"
    "Input: The lawyer provided prompt responses and practical solutions throughout the negotiation process.\n"
    "Output:"
)
prompt = "Instruction: Segment the following feedback into a summary.\nInput: The lawyer provided prompt responses and practical solutions throughout the negotiation process.\nOutput:"


inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    temperature=1.0,  # ⬆️ more diversity
    top_p=0.95,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print("\nRaw decode:")
print(tokenizer.decode(outputs[0]))
print("\n🧾 Model output:\n", tokenizer.decode(outputs[0], skip_special_tokens=True))


Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 2861.44 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
/Users/sathishkumarchandran/IdeaProjects/llm/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,10.257300
20,7.008500
30,4.337200
40,2.938300



Raw decode:
Instruction: Segment the following feedback into a summary.
Input: The lawyer provided prompt responses and practical solutions throughout the negotiation process.
Output:<|im_end|>

🧾 Model output:
 Instruction: Segment the following feedback into a summary.
Input: The lawyer provided prompt responses and practical solutions throughout the negotiation process.
Output:
